In [ ]:
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd
from openpyxl import Workbook
import pytz

file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/FBK/PRE-SERIE_FBK_Vendor_16x16_IV.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
row = array('i', [0])
column = array('i', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)
#tree.Branch("I_GR", "std::vector<float>", IBACK)

txt_files = glob.glob("/Users/icosivi/cernbox/MTD/QAQC/FBKdata/UFSD_LF/DEV_16x16_new/*.txt")

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_sensor-col-row.xlsx')

counter = 0

for txt in txt_files:
    #print(txt)
    V.clear()
    #IBACK.clear()

    with open(txt,"r") as t:
        lines_list = t.readlines()
        header_v = lines_list[0]
        
        header_v.strip()
        hv = re.split("\s+", header_v)
        
        #for m in hv[6:]:
        for m in hv[8:]:    
            if not m.isspace():
                if m:
                    print(m)
                    V.push_back( abs(float(m)) )

        lines = lines_list[1:]
        
        for line in lines:
            line.strip()
            IBACK.clear()
            ll = re.split( '\s+', line)
            #print(ll[5])
            #if ll[4] == "I_BACK":
            #if ll[5] == "I_GR":
            #if ll[5] == "I_BACK[A]":
            if ll[5] == "I_COL[A]":
                event[0] = counter
                wafer[0] = int( ll[0] )
                column[0] = int( ll[1] )
                row[0] = int( ll[2] )
                sensor[0] = int(wb[(wb['Row'] == row[0]) & (wb['Column'] == column[0])]['SerialNumber'].iloc[0])
                  
                '''
                if ll[4] == "I_BACK":
                    type_def = re.split("_", ll[3])
                    #print(type_def[0])
                    if(type_def[2]=='PIN'):
                        type[0] = 0
                        event[0] = counter
                        wafer[0] = int( ll[0] )
                        column[0] = int( ll[1] )
                        row[0] = int( ll[2] )
                    elif(type_def[2]=='PAD'):
                        type[0] = 1
                        event[0] = counter
                        wafer[0] = int( ll[0] )
                        column[0] = int( ll[1] )
                        row[0] = int( ll[2] )
                '''
                
                '''
                sensor_types = re.split( '_|-', ll[3])
                ggtype = re.findall(r'\d+',sensor_types[1])
                gr_type[0] = int(ggtype[0])
                for s in sensor_types[1]:
                    if s.isdigit():    
                        gr_type[0] = int( s )
                
                for p in sensor_types[2]:
                    if p.isdigit():    
                        grn[0] = int( p )
                for z in sensor_types[2]:
                    if z.isdigit():    
                        grt[0] = int( z )
                    if z == "S":
                        grt[0] = int(2)
                '''
                #for q in ll[7:]:
                for q in ll[6:]:
                    if not q.isspace():
                        if q:
                            IBACK.push_back( abs(float(q)) )
                            #if counter==2:
                                #print(q)          
                #if not any(x == "PIN" for x in sensor_types):
                tree.Fill()
                counter += 1

tree.Write()
file.Write()
file.Close()

In [ ]:
# producer of the xls to register components on the database for the 16x16
xl_filename="FBK_16x16_PRE-SERIES"
ww = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

for i in range(n_serialn):
    wws["A%i" %(serial_num_counter+1)] = 'PRE'+str(serial_num)
    wws["B%i" %(serial_num_counter+1)] = 'FBK-LFoundry'
    wws["C%i" %(serial_num_counter+1)] = batch
    wws["D%i" %(serial_num_counter+1)] = Wafer
    wws["E%i" %(serial_num_counter+1)] = "16x16"
    wws["F%i" %(serial_num_counter+1)] = Row
    wws["G%i" %(serial_num_counter+1)] = Column
    
    sensor_number = int(wb[(wb['Row'] == Row) & (wb['Column'] == Column)]['SerialNumber'].iloc[0])
    wws["H%i" %(serial_num_counter+1)] = sensor_number

save_path = '/Users/icosivi/Desktop/PRE-SERIE/FBK/'
wb.save(save_path+xl_filename+".xlsx")